# Detecting entanglement dimensionality with isotypic projective measurements

**Companion notebook to the thesis.** The goal is to use *isotypic projective measurements*
(isotypic / Young projectors of the symmetric group) to detect entanglement dimensionality
— Schmidt rank, Schmidt number, and tensor rank — across partitions of multipartite states.

The central object is the isotypic projector $\Pi_\lambda$ on $(\mathbb{C}^d)^{\otimes k}$.
Its key property is
$$\big\|(\Pi_\lambda\otimes I_B)\,|\psi_{AB}\rangle^{\otimes k}\big\|^2 = f^\lambda\, s_\lambda(s_1^2,\dots,s_r^2),$$
where the $s_i$ are the Schmidt coefficients of $|\psi_{AB}\rangle$, $f^\lambda=\dim V^\lambda$,
and $s_\lambda$ is the Schur polynomial. This norm vanishes **iff** the Schmidt rank is below
the height $h(\lambda)$, and the construction extends to mixed states through a
convex-roof / semidefinite-programming relaxation.

The notebook implements and numerically checks each step: characters and isotypic projectors of
$S_k$, the vanishing condition on the computational basis, the closed-form projected norm, the
Schur-polynomial identity, and the Schmidt-number SDP.

> Runs in a **SageMath** kernel; the SDP section additionally needs `picos` and `cvxopt`.

## 1. Mathematical background

### 1.1 Pure states, measurement, tensor products, gates

The smallest unit of quantum information is the **qubit**, a unit vector
$|\psi\rangle\in\mathbb{C}^2$; more generally a **qudit** is a unit vector
$|\psi\rangle\in\mathbb{C}^d$ in a Hilbert space $\mathcal{H}=\mathbb{C}^d$. The conjugate is
$\langle\psi|=(|\psi\rangle)^\dagger$, the inner product of $u,v$ is $\langle u|v\rangle$, and the
projector onto $|\psi\rangle$ is $|\psi\rangle\langle\psi|$.

**Projective measurement.** For a Hermitian operator $O$ with eigenbasis $\{o_i,|v_i\rangle\}_i$
and a state $|\psi\rangle$, the probability of outcome $o_i$ is
$p(o_i)=\langle\psi|v_i\rangle\langle v_i|\psi\rangle$, after which the state collapses to
$|v_i\rangle$.

**Composite systems.** The state space of a composite system is the tensor product of the parts:
for $|\psi\rangle\in\mathbb{C}^n$, $|\phi\rangle\in\mathbb{C}^m$ we have
$|\psi\rangle\otimes|\phi\rangle\in\mathbb{C}^{nm}$. Not every vector in $\mathbb{C}^{nm}$
factorises — the origin of entanglement.

**Gates.** Quantum gates are unitaries $U$ ($U^\dagger U=I$) acting on one or more qudits and
returning pure states.

### 1.2 Entanglement, Schmidt rank and Schmidt number

A bipartite pure state $|\psi_{AB}\rangle\in\mathcal{H}^A\otimes\mathcal{H}^B$ is **separable** if
it factorises and **entangled** otherwise. Every bipartite pure state has a Schmidt decomposition
$$|\psi_{AB}\rangle=\sum_{i=1}^r s_i\,|v_i\rangle_A\otimes|w_i\rangle_B,$$
with orthonormal $\{|v_i\rangle\}$, $\{|w_i\rangle\}$ and $s_i>0$. The minimal $r$ is the **Schmidt
rank** $\mathrm{SR}(|\psi_{AB}\rangle)$ (the rank of the coefficient matrix), and the $s_i$ with
$\sum_i s_i^2=1$ are the **Schmidt coefficients** — the quantities the rest of this notebook works
with.

**Mixed states.** A mixed state is a convex mixture $\rho=\sum_i p_i|\psi_i\rangle\langle\psi_i|$.
The generalisation of Schmidt rank is the **Schmidt number**
$$\mathrm{SN}(\rho)=\min_{\mathcal{D}(\rho)}\ \max_i\ \mathrm{SR}(|\psi_i\rangle),$$
minimised over all decompositions $\mathcal{D}(\rho)=\{p_i,|\psi_i\rangle:\rho=\sum_i p_i|\psi_i\rangle\langle\psi_i|\}$.
Deciding separability — and the Schmidt number more generally — is computationally hard, which
motivates the projector-based detectors below.

**Reduced states.** With orthonormal bases $\{|e^A_i\rangle\}$, $\{|e^B_j\rangle\}$, the partial
trace gives
$$\rho_B=\operatorname{tr}_A(\rho_{AB})=\sum_i(\langle e^A_i|\otimes I_B)\,\rho_{AB}\,(|e^A_i\rangle\otimes I_B).$$

### 1.3 Representation theory essentials

A **representation** of a finite group $G$ on $V$ is a homomorphism $G\to GL(V)$ sending each
element to an invertible matrix while respecting the group law. It is **irreducible** (an *irrep*)
if it has no non-trivial invariant subspace, and every representation of a finite group decomposes
as a direct sum of irreps.

The **group algebra** $A(G)$ is the complex span of the group elements, $a=\sum_{g\in G}a(g)\,g$.
Representations extend linearly and multiplicatively to $A(G)$, with $V(e)=I$ and, for unitary
reps, $V(a^*)=V(a)^\dagger$. **Minimal projections** $p\in A(G)$ ($p^2=p$, indecomposable)
correspond one-to-one with irreducible subspaces; two are either equivalent ($upv=q$) or disjoint
($puq=0$).

**Characters.** The character of $V$ is $\chi(g)=\operatorname{tr}V(g)$; equivalent representations
share a character. With the inner product
$$\langle f,h\rangle_G=\frac1{|G|}\sum_{g\in G}f(g)\,\overline{h(g)},$$
irreducible characters are orthonormal, $\langle\chi_{\lambda_1},\chi_{\lambda_2}\rangle_G=\delta_{\lambda_1\lambda_2}$.
Characters are constant on conjugacy classes — the fact that makes the projector formula below
efficient.

## 2. The symmetric group, Young diagrams and isotypic projectors

### The group $S_k$ and its commutant

Let $S_k$ act on $(\mathbb{C}^d)^{\otimes k}$ by permuting the tensor factors (in cycle notation
$\pi_{(a\,b\,c\,d)}$ rotates the indicated factors). Independently, $U\in U(d)$ acts diagonally,
$U:|e_{i_1}\rangle\otimes\cdots\otimes|e_{i_k}\rangle\mapsto U|e_{i_1}\rangle\otimes\cdots\otimes U|e_{i_k}\rangle$.
On every basis vector the two actions commute, hence $S_k$ and $U(d)$ commute on all of
$(\mathbb{C}^d)^{\otimes k}$ — the starting point of Schur–Weyl duality.

### Young diagrams, tableaux and symmetrizers

A **partition** $\lambda\vdash k$ (non-increasing, summing to $k$) is drawn as a **Young diagram**
with $\lambda_i$ boxes in row $i$. A **Young tableau** fills the boxes; a **standard** tableau (SYT)
uses each of $1,\dots,k$ once, increasing along rows and down columns. The number of SYT of shape
$\lambda$ is the hook-length formula
$$f^\lambda=\frac{k!}{\prod_{\square\in\lambda}h(\square)}.$$
To a tableau $T$ one associates the row- and column-symmetrizers
$$r(T)=\sum_{\pi\in R(T)}\pi,\qquad c(T)=\sum_{\pi\in C(T)}\operatorname{sign}(\pi)\,\pi,$$
and the **Young symmetrizer** $e_T=r(T)\,c(T)$.

> **Theorem (Christandl 1.14).** For a standard tableau $T$ of shape $\lambda$,
> $\tfrac{f^\lambda}{k!}e_T$ is a minimal projection onto the irrep $V^\lambda$ of $S_k$, with
> $\dim V^\lambda=f^\lambda$. The $V^\lambda$ for $\lambda\vdash k$ form a complete set of irreps,
> and $\;\mathbb{C}[S_k]\cong\bigoplus_{\lambda\vdash k}(V^\lambda)^{\oplus f^\lambda}.$

### The isotypic projector

Summing the minimal projections within one isotypic component gives the **isotypic projector** onto
the $\lambda$-sector,
$$\Pi_\lambda=\frac{\chi_\lambda(\mathrm{id})}{k!}\sum_{\sigma\in S_k}\chi_\lambda(\sigma^{-1})\,\sigma=\frac{f^\lambda}{k!}\sum_{\sigma\in S_k}\chi_\lambda(\sigma)\,\sigma.$$
Because $\chi_\lambda$ is constant on conjugacy classes, the code evaluates it once per class rather
than once per group element.

In [1]:
import math
import itertools
from itertools import product as iproduct

import numpy as np

import sage.all as sage
from sage.rings.rational_field import QQ
from sage.combinat.sf.sf import SymmetricFunctions

# Ring of symmetric functions over the rationals, with the two bases we need.
_Sym = SymmetricFunctions(QQ)
_s   = _Sym.schur()       # Schur basis      s_lambda
_p   = _Sym.powersum()    # power-sum basis  p_lambda


### 2.1 Characters of $S_k$ on its conjugacy classes

We first extract the irreducible character $\chi_\lambda$ from Sage's character table, evaluated on
every conjugacy class — the data that enters the projector formula.

In [2]:
def character_on_conjugacy_classes(k: int, lam: list[int]) -> list[tuple[list, object]]:
    """Irreducible character chi_lambda of S_k evaluated on every conjugacy class.

    Parameters
    ----------
    k   : size of the symmetric group S_k.
    lam : partition of k labelling the irreducible representation V^lambda.

    Returns
    -------
    A list of (class_elements, chi_value) pairs, one per conjugacy class:
    ``class_elements`` lists the permutations in the class and ``chi_value`` is
    chi_lambda evaluated there (constant on the class).
    """
    assert sum(lam) == k
    lam = sorted(lam, reverse=True)              # normalise to a non-increasing partition

    Sk = sage.SymmetricGroup(k)
    char_table = Sk.character_table()            # rows: irreps, columns: conjugacy classes
    classes = Sk.conjugacy_classes()

    # Sage orders character-table rows by partitions in reverse-lexicographic order.
    partitions_ordered = list(reversed(sage.Partitions(k).list()))
    row_idx = partitions_ordered.index(lam)

    return [(cl.list(), char_table[row_idx][j]) for j, cl in enumerate(classes)]


# Sanity check: reproduce the known character table of S_3.
for lam in sage.Partitions(3).list():
    result = character_on_conjugacy_classes(3, lam)
    print(f"\nlambda = {lam}:")
    for elements, chi in result:
        print(f"  sigma = {elements[0]}, chi = {chi}")



lambda = [3]:
  sigma = (), chi = 1
  sigma = (2,3), chi = 1
  sigma = (1,2,3), chi = 1

lambda = [2, 1]:
  sigma = (), chi = 2
  sigma = (2,3), chi = 0
  sigma = (1,2,3), chi = -1

lambda = [1, 1, 1]:
  sigma = (), chi = 1
  sigma = (2,3), chi = -1
  sigma = (1,2,3), chi = 1


### 2.2 The permutation action on $(\mathbb{C}^n)^{\otimes k}$

Each $\sigma\in S_k$ is represented as an $n^k\times n^k$ permutation matrix reordering the tensor
factors. We build it column by column on the computational basis.

In [3]:
def permutation_matrix_on_tensor_power(sigma, n: int, k: int) -> np.ndarray:
    """Matrix of a permutation sigma in S_k acting on (C^n)^{otimes k}.

    The symmetric group permutes the k tensor factors:
        sigma |e_{i_1}> (x) ... (x) |e_{i_k}>  =  |e_{i_{sigma^{-1}(1)}}> (x) ... .
    The result is an (n^k) x (n^k) permutation matrix (a single 1 per column).

    Parameters
    ----------
    sigma : element of Sage's SymmetricGroup(k).
    n     : local dimension (single-factor space V = C^n).
    k     : number of tensor factors.
    """
    basis = list(iproduct(range(n), repeat=k))     # all index tuples (i_1, ..., i_k)
    index = {b: i for i, b in enumerate(basis)}    # tuple -> row/column position

    sigma_inv = sigma.inverse()
    dim = n ** k
    M = np.zeros((dim, dim), dtype=complex)

    for col_idx, basis_vec in enumerate(basis):
        # Output factor j reads input factor sigma^{-1}(j) (Sage is 1-indexed).
        new_basis = tuple(basis_vec[sigma_inv(j + 1) - 1] for j in range(k))
        row_idx = index[new_basis]
        M[row_idx, col_idx] = 1.0

    return M


### 2.3 Building $\Pi_\lambda$

We assemble $\Pi_\lambda=\frac{f^\lambda}{k!}\sum_\sigma\chi_\lambda(\sigma)\,\sigma$, looping over
conjugacy classes (constant character) and summing the permutation matrices.

In [4]:
def isotypic_projector(lam: list[int], n: int, k: int) -> np.ndarray:
    """Isotypic projector Pi_lambda onto the lambda-component of (C^n)^{otimes k}.

    Implements
        Pi_lambda = (f^lambda / k!) * sum_{sigma in S_k} chi_lambda(sigma) sigma,
    where f^lambda = chi_lambda(id) = dim V^lambda. Since chi_lambda is constant
    on conjugacy classes, we read it once per class and reuse it for all members.

    Parameters
    ----------
    lam : partition of k labelling the target irrep V^lambda.
    n   : local dimension.
    k   : number of tensor factors (must equal sum(lam)).
    """
    assert sum(lam) == k
    lam = sorted(lam, reverse=True)

    Sk = sage.SymmetricGroup(k)
    char_table = Sk.character_table()
    classes = Sk.conjugacy_classes()

    # Locate the character-table row for lambda (reverse-lex ordering, as above).
    partitions_ordered = list(reversed(sage.Partitions(k).list()))
    row_idx = partitions_ordered.index(lam)

    # f^lambda = chi_lambda(identity) is the dimension of the irrep.
    id_idx = next(j for j, cl in enumerate(classes)
                  if cl.representative().is_one())
    d_lam = int(char_table[row_idx][id_idx])

    dim = n ** k
    Pi = np.zeros((dim, dim), dtype=complex)

    # Accumulate chi_lambda(sigma) * sigma over the whole group, class by class.
    for j, cl in enumerate(classes):
        chi = complex(char_table[row_idx][j])      # character is constant on the class
        for sigma in cl:
            M = permutation_matrix_on_tensor_power(sigma, n, k)
            Pi += chi * M

    Pi *= d_lam / sage.factorial(k)                # overall prefactor f^lambda / k!
    return Pi


### 2.4 Sanity checks

A valid isotypic projector must be Hermitian and idempotent ($\Pi_\lambda^2=\Pi_\lambda$), have
integer trace $f^\lambda\times(\text{multiplicity})$, and the projectors over all $\lambda\vdash k$
must resolve the identity.

In [5]:
n, k = 2, 3  # qubit example: V = C^2 with k = 3 tensor factors

for lam in sage.Partitions(k).list():
    Pi = isotypic_projector(lam, n, k)

    # A projector must be idempotent ...
    assert np.allclose(Pi @ Pi, Pi), f"{lam}: not idempotent"
    # ... and Hermitian.
    assert np.allclose(Pi, Pi.conj().T), f"{lam}: not Hermitian"

    # trace(Pi_lambda) = f^lambda * (multiplicity of V^lambda in the tensor space).
    print(f"lambda={lam}, trace={np.trace(Pi).real:.1f}")

# The isotypic projectors over all lambda |- k resolve the identity.
total = sum(isotypic_projector(lam, n, k)
            for lam in sage.Partitions(k).list())
assert np.allclose(total, np.eye(n**k))
print("Projectors sum to identity \u2713")


lambda=[3], trace=4.0
lambda=[2, 1], trace=4.0
lambda=[1, 1, 1], trace=0.0
Projectors sum to identity ✓


## 3. Computational basis vectors and the vanishing condition

### The occupation type $\nu$

Fix the standard basis $\{|e_i\rangle\}$ of $\mathbb{C}^d$. A basis vector
$|e_{i_1}\rangle\otimes\cdots\otimes|e_{i_k}\rangle$ is summarised by its **occupation type**
$\nu=(\nu_1,\dots,\nu_{h(\nu)})$: the sorted label multiplicities, $\nu_1$ the count of the most
frequent label. The number of distinct labels $h(\nu)$ is the **height** of $\nu$.

### Vanishing of $\Pi_\lambda$ on basis vectors

Say $\lambda$ **majorises** $\nu$ when $\sum_{i\le q}\lambda_i\ge\sum_{i\le q}\nu_i$ for all $q$. If
$\lambda$ does **not** majorise $\nu$ — which necessarily holds when $h(\nu)<h(\lambda)$ — then the
basis vector is annihilated,
$$\Pi_\lambda\big(|e_{i_1}\rangle\otimes\cdots\otimes|e_{i_k}\rangle\big)=0.$$
A sector $\lambda$ can only be populated when the state has enough distinct components (Christandl's
vanishing condition).

### 3.1 Basis kets

Helper to materialise a computational basis vector $|i_1,\dots,i_k\rangle$ as an explicit vector in
$(\mathbb{C}^n)^{\otimes k}\cong\mathbb{C}^{n^k}$.

In [6]:
import numpy as np
from itertools import product as iproduct

def ket(indices: list[int], n: int) -> np.ndarray:
    """
    Convert a basis vector in ket notation to a vector in (C^n)^{otimes k}.
    
    indices: tuple of ints, e.g. (2, 0, 3) for |2,0,3>
    n: local dimension, e.g. n=4 for C^4
    
    Example: ket((2,0,3), n=4) -> unit vector in C^{4^3} = C^64
    """
    k = len(indices)
    assert all(0 <= i < n for i in indices), f"All indices must be in range [0, {n-1}]"

    basis = list(iproduct(range(n), repeat=k))
    index = {b: i for i, b in enumerate(basis)}

    v = np.zeros(n**k, dtype=complex)
    v[index[tuple(indices)]] = 1.0
    return v

# |2,0,3> in (C^4)^{otimes 3}
v = ket([2,0,3], n=4)
print(v.shape)   # (64,)
print(v.sum())   # 1.0 — exactly one nonzero entry

# Recover the index
print(np.argmax(v))  # 2*16 + 0*4 + 3 = 35

(64,)
(1+0j)
35


### 3.2 Projecting a ket and the orthogonality of sectors

We project a basis vector onto a chosen sector and verify idempotence on the image, plus that the
symmetric, mixed and antisymmetric components are mutually orthogonal and reconstruct the original
vector.

In [7]:
v = ket([0, 0, 3], n=4)
Pi = isotypic_projector([3], n=4, k=3)

projected = Pi @ v

# Norm of the projected vector — how much of |2,0,3> lives in this subspace
print(np.linalg.norm(projected))

# Applying the projector twice should give the same result
print(np.allclose(Pi @ projected, projected))  # True

# Projectors onto different sectors are orthogonal
Pi_sym  = isotypic_projector([3],     n=4, k=3)
Pi_mix  = isotypic_projector([2,1],   n=4, k=3)
Pi_anti = isotypic_projector([1,1,1], n=4, k=3)

v_sym  = Pi_sym  @ v
v_mix  = Pi_mix  @ v
v_anti = Pi_anti @ v

print(v_anti)

# These three components are orthogonal and reconstruct v
print(np.allclose(v_sym + v_mix + v_anti, v))           # True
print(np.allclose(np.dot(v_sym.conj(), v_mix), 0))      # True

0.5773502691896257
True
[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
True
True


### 3.3 The occupation type and the direct vanishing test

`occupation_type(v, n)` returns the type $\nu$. `projected_norm(lam, n, v)` builds $\Pi_\lambda$
explicitly and returns the number $\lVert\Pi_\lambda|v\rangle\rVert$, and `projects_to_zero` is just
the boolean predicate "that norm is below tolerance". The test below prints, for a fixed state, each
sector's height, its **actual** projected norm, whether it is zero, and the **expected** outcome,
with a PASS/FAIL flag. (The sufficient rule shown here is $h(\nu)<h(\lambda)\Rightarrow$ vanish; the
exact iff criterion follows in 3.4.)

In [8]:
def occupation_type(v: list[int], n: int) -> list[int]:
    """Occupation type nu of a basis vector |e_{v_1}, ..., e_{v_k}>.

    Counts how often each of the n local labels occurs in ``v`` and returns
    those multiplicities sorted non-increasingly, with zeros dropped.
    Example: occupation_type([0, 0, 1], n=2) -> [2, 1].
    """
    counts = [0] * n              # n could equivalently be inferred as max(v) + 1
    for label in v:
        counts[label] += 1
    counts.sort(reverse=True)
    return [c for c in counts if c != 0]


def projected_norm(lam: list[int], n: int, v: list[int]) -> float:
    """|| Pi_lambda |v> ||  for a single computational basis vector |v>.

    Builds the isotypic projector explicitly and applies it to |v>.
    """
    k = sum(lam)
    Pi = isotypic_projector(lam, n, k)
    return float(np.linalg.norm(Pi @ ket(v, n)))


def projects_to_zero(lam: list[int], n: int, v: list[int], tol: float = 1e-5) -> bool:
    """True iff Pi_lambda annihilates the basis vector |v> (projected norm < tol)."""
    return projected_norm(lam, n, v) < tol


In [9]:
def run_vanishing_tests(n: int, v: list[int],
                        cases: list[tuple[list[int], bool]]) -> None:
    """Pretty-print a vanishing test for a fixed basis vector |v>.

    cases : list of (lambda, expected_to_vanish) pairs.
    For each sector lambda it shows the height h(lambda), the actual projected
    norm || Pi_lambda |v> ||, whether that is zero, the expected outcome, and a
    PASS/FAIL flag.
    """
    nu = occupation_type(v, n)
    k = len(v)
    print(f"State        |v> = |{','.join(map(str, v))}>   in (C^{n})^(x){k}")
    print(f"Occupation   nu  = {nu}   (height h(nu) = {len(nu)})")
    print(f"Vanishes when lambda cannot dominate nu  (always if h(lambda) > h(nu))\n")

    print(f"  {'lambda':<16}{'h(lam)':>6}   {'||Pi|v>||':>10}   {'got':>8}   {'expected':>8}   status")
    print("  " + "-" * 64)
    for lam, expect_zero in cases:
        norm = projected_norm(lam, n, v)
        is_zero  = norm < 1e-5
        got      = "ZERO" if is_zero else "nonzero"
        expected = "ZERO" if expect_zero else "nonzero"
        status   = "PASS" if is_zero == expect_zero else "FAIL  <<<"
        print(f"  {str(lam):<16}{len(lam):>6}   {norm:>10.4f}   "
              f"{got:>8}   {expected:>8}   {status}")


# (C^2)^(x)3 : only two distinct labels are available, so any sector whose
# height exceeds 2 must vanish.
run_vanishing_tests(
    n=2,
    v=[0, 0, 1],
    cases=[
        ([3],       False),   # fully symmetric          -> populated
        ([2, 1],    False),   # mixed symmetry           -> populated
        ([1, 1, 1], True),    # antisymmetric needs 3 distinct labels -> vanishes
    ],
)


State        |v> = |0,0,1>   in (C^2)^(x)3
Occupation   nu  = [2, 1]   (height h(nu) = 2)
Vanishes when lambda cannot dominate nu  (always if h(lambda) > h(nu))

  lambda          h(lam)    ||Pi|v>||        got   expected   status
  ----------------------------------------------------------------
  [3]                  1       0.5774    nonzero    nonzero   PASS
  [2, 1]               2       0.8165    nonzero    nonzero   PASS
  [1, 1, 1]            3       0.0000       ZERO       ZERO   PASS


In [10]:
# (C^3)^(x)6 with v = [0,0,0,1,1,2] : occupation type nu = [3,2,1], so h(nu) = 3.
# Sectors of height <= 3 can be populated; height 4 and 5 must vanish.
run_vanishing_tests(
    n=3,
    v=[0, 0, 0, 1, 1, 2],
    cases=[
        ([6],             False),   # h(lam) = 1
        ([4, 2],          False),   # h(lam) = 2
        ([3, 3],          False),   # h(lam) = 2
        ([3, 2, 1],       False),   # h(lam) = 3 = h(nu)
        ([2, 2, 1, 1],    True),    # h(lam) = 4 > h(nu) = 3  -> vanishes
        ([2, 1, 1, 1, 1], True),    # h(lam) = 5 > h(nu) = 3  -> vanishes
    ],
)


State        |v> = |0,0,0,1,1,2>   in (C^3)^(x)6
Occupation   nu  = [3, 2, 1]   (height h(nu) = 3)
Vanishes when lambda cannot dominate nu  (always if h(lambda) > h(nu))

  lambda          h(lam)    ||Pi|v>||        got   expected   status
  ----------------------------------------------------------------
  [6]                  1       0.1291    nonzero    nonzero   PASS
  [4, 2]               2       0.5477    nonzero    nonzero   PASS
  [3, 3]               2       0.2887    nonzero    nonzero   PASS
  [3, 2, 1]            3       0.5164    nonzero    nonzero   PASS
  [2, 2, 1, 1]         4       0.0000       ZERO       ZERO   PASS
  [2, 1, 1, 1, 1]      5       0.0000       ZERO       ZERO   PASS


### 3.4 Majorisation form of the condition

The exact criterion is: $\Pi_\lambda$ annihilates $|v\rangle$ **iff** $\lambda$ does *not* majorise
$\nu$. `majorizes(lam, nu)` implements the prefix-sum test ($\lambda$ majorises $\nu$ when every
cumulative sum of $\lambda$ is $\ge$ that of $\nu$) and `cumulative_sums` exposes those prefix sums.
The test prints both prefix-sum vectors, the majorisation verdict, the prediction it implies, and the
actual projected norm.

In [11]:
def cumulative_sums(p: list[int]) -> list[int]:
    """Prefix sums [p_1, p_1+p_2, ...]; the data compared in the majorisation test."""
    out, running = [], 0
    for x in p:
        running += x
        out.append(running)
    return out


def majorizes(lam: list[int], nu: list[int]) -> bool:
    """True iff lambda majorises nu: every prefix sum of lambda is >= that of nu.

    Both partitions sum to the same k; the shorter prefix-sum vector is padded
    with k so the two have equal length before the element-wise comparison.
    """
    assert sum(lam) == sum(nu)
    k = sum(lam)
    lam_c = cumulative_sums(lam)
    nu_c  = cumulative_sums(nu)
    length = max(len(lam_c), len(nu_c))
    lam_c += [k] * (length - len(lam_c))
    nu_c  += [k] * (length - len(nu_c))
    return all(a >= b for a, b in zip(lam_c, nu_c))


In [12]:
# Exact criterion:  Pi_lambda |v> = 0  <=>  lambda does NOT majorize nu.
# For each sector we print the prefix sums of lambda and nu, the majorization
# verdict, the prediction it implies, and the actual projected norm.
n = 3
v = [0, 0, 0, 1, 1, 2]
nu = occupation_type(v, n)

print(f"State |v> = |{','.join(map(str, v))}>   nu = {nu}   prefix(nu) = {cumulative_sums(nu)}\n")
print(f"  {'lambda':<16}{'prefix(lam)':<18}{'maj?':>5}   {'predicted':>9}   {'||Pi|v>||':>10}   status")
print("  " + "-" * 72)

for lam in [[6], [4, 2], [3, 3], [3, 2, 1], [2, 2, 1, 1], [2, 1, 1, 1, 1]]:
    maj = majorizes(lam, nu)               # does lambda majorize nu?
    predicted_zero = not maj               # it vanishes exactly when it does not
    norm = projected_norm(lam, n, v)
    actual_zero = norm < 1e-5
    status = "PASS" if predicted_zero == actual_zero else "FAIL  <<<"
    print(f"  {str(lam):<16}{str(cumulative_sums(lam)):<18}"
          f"{('yes' if maj else 'no'):>5}   "
          f"{('ZERO' if predicted_zero else 'nonzero'):>9}   {norm:>10.4f}   {status}")


State |v> = |0,0,0,1,1,2>   nu = [3, 2, 1]   prefix(nu) = [3, 5, 6]

  lambda          prefix(lam)        maj?   predicted    ||Pi|v>||   status
  ------------------------------------------------------------------------
  [6]             [6]                 yes     nonzero       0.1291   PASS
  [4, 2]          [4, 6]              yes     nonzero       0.5477   PASS
  [3, 3]          [3, 6]              yes     nonzero       0.2887   PASS
  [3, 2, 1]       [3, 5, 6]           yes     nonzero       0.5164   PASS
  [2, 2, 1, 1]    [2, 4, 5, 6]         no        ZERO       0.0000   PASS
  [2, 1, 1, 1, 1] [2, 3, 4, 5, 6]      no        ZERO       0.0000   PASS


## 4. The projected norm in terms of Schmidt coefficients

For a basis vector, $\langle v_\nu|\Pi_\lambda|v_\nu\rangle$ depends only on the occupation type
$\nu$. Using $\Pi_\lambda^\dagger\Pi_\lambda=\Pi_\lambda$, Frobenius reciprocity, and Young's rule
$M^\nu\cong\bigoplus_\lambda K_{\lambda\nu}V^\lambda$ (with $K_{\lambda\nu}$ the Kostka numbers),
$$\|\Pi_\lambda|v_\nu\rangle\|^2=\langle v_\nu|\Pi_\lambda|v_\nu\rangle=\frac{f^\lambda K_{\lambda\nu}}{\binom{k}{\nu_1,\dots,\nu_r}},$$
where $K_{\lambda\nu}=0$ exactly when $\nu\succ\lambda$ (e.g. when $h(\nu)<h(\lambda)$).

For a general entangled state $|\psi_{AB}\rangle=\sum_{i=1}^r s_i|a_i\rangle_A|b_i\rangle_B$ with
orthonormal $\{|a_i\rangle\}$, summing over occupation types and weighting by the squared Schmidt
coefficients gives
$$\|(\Pi_\lambda\otimes I)\,|\psi_{AB}\rangle^{\otimes k}\|^2=f^\lambda\sum_{\nu\vdash k}K_{\lambda\nu}\,m_\nu(s_1^2,\dots,s_r^2),$$
with $m_\nu$ the monomial symmetric polynomial. In particular this norm is $0$ **iff**
$\mathrm{SR}(|\psi_{AB}\rangle)<h(\lambda)$.

In Section 5 the Kostka sum $\sum_\nu K_{\lambda\nu}\,m_\nu$ is identified as a Schur polynomial,
giving a single closed form valid for every $\lambda$.

## 5. The Schur-polynomial formula

The sum $\sum_{\nu\vdash k}K_{\lambda\nu}\,m_\nu(s_1^2,\dots,s_r^2)$ is exactly the **Schur
polynomial** $s_\lambda$ at the squared Schmidt coefficients. Hence the projected norm has the
compact closed form
$$\boxed{\ \|(\Pi_\lambda\otimes I)\,|\psi_{AB}\rangle^{\otimes k}\|^2=f^\lambda\, s_\lambda(s_1^2,s_2^2,\dots,s_r^2)\ }$$
valid for **every** partition $\lambda$. Since $s_\lambda$ vanishes when its number of variables $r$
falls below $h(\lambda)$, this recovers the criterion $\|\cdot\|^2=0\iff\mathrm{SR}<h(\lambda)$ while
the numerical value encodes the whole Schmidt spectrum. Below we compute $f^\lambda$ via the
hook-length formula and $s_\lambda$ via Sage, then check the identity against the explicit
projector.

In [13]:
def f_lambda(lam: list[int]) -> int:
    """Number of standard Young tableaux of shape lam, via the hook-length formula f^lambda = k! / prod h(u)."""
    lam = sorted(lam, reverse=True)
    k = sum(lam)
    hook_product = 1
    for i, row_len in enumerate(lam):
        for j in range(row_len):
            arm = row_len - j - 1                          # cells to the right
            leg = sum(1 for r in lam[i + 1:] if r > j)    # cells below
            hook_product *= arm + leg + 1
    return math.factorial(k) // hook_product

# Sanity checks against known values
assert f_lambda([3])     == 1   # single row
assert f_lambda([1,1,1]) == 1   # single column
assert f_lambda([2,1])   == 2
assert f_lambda([3,2,1]) == 16
print("f_lambda checks passed ✓")

f_lambda checks passed ✓


In [14]:
def norm_schur(lam: list[int], sis: list[float]) -> float:
    """||Pi_lambda psi_AB^{otimes k}||^2 = f^lambda * s_lambda(s1^2, ..., sr^2).

    lam: Young frame partition (any order, will be sorted)
    sis: Schmidt coefficients with sum(si^2) = 1
    """
    sisq = [si**2 for si in sis]
    return f_lambda(lam) * schur_polynomial(lam, sisq)


# Quick checks for schur_polynomial
# s_{[1]}(x) = x_1 + x_2  →  sum = 1
# Patch: allow Python floats in evaluation by working over RDF
def schur_polynomial(lam: list[int], x: list[float]) -> float:
    lam = sorted(lam, reverse=True)
    r = len(x)
    poly = _s[lam].expand(r).change_ring(sage.RDF)
    return float(poly(*[sage.RDF(xi) for xi in x]))

assert abs(schur_polynomial([1], [0.3, 0.7]) - 1.0) < 1e-10

# s_{[2]} = h_2: 0.25 + 0.25 + 0.25 = 0.75 at (0.5, 0.5)
assert abs(schur_polynomial([2], [0.5, 0.5]) - 0.75) < 1e-10

# s_{[1,1]} = e_2: 0.5 * 0.5 = 0.25 at (0.5, 0.5)
assert abs(schur_polynomial([1, 1], [0.5, 0.5]) - 0.25) < 1e-10

# height > r  →  0
assert abs(schur_polynomial([1, 1, 1], [0.5, 0.5])) < 1e-10

print("schur_polynomial checks passed ✓")

schur_polynomial checks passed ✓


In [15]:
def norm_sq_projected_general(lam: list[int], sis: list[float]) -> float:
    """Ground-truth ||Pi_lam psi_AB^{otimes k}||^2 for ANY partition lam,
    obtained by building Pi_lam explicitly and summing its weighted diagonal."""
    r = len(sis)
    sisq = [si ** 2 for si in sis]
    k = sum(lam)
    Pi = isotypic_projector(lam, n=r, k=k)
    basis = list(iproduct(range(r), repeat=k))
    basis_index = {b: i for i, b in enumerate(basis)}
    total = 0.0
    for idx_tuple in basis:
        weight = math.prod(sisq[i] for i in idx_tuple)
        if weight == 0:
            continue
        total += weight * Pi[basis_index[idx_tuple], basis_index[idx_tuple]].real
    return total


def assert_close(a: float, b: float, tol: float = 1e-6, label: str = "") -> None:
    err = abs(a - b)
    status = "PASS" if err < tol else "FAIL"
    print(f"[{status}] {label}")
    if err >= tol:
        print(f"        got {a:.8f}, expected {b:.8f}, diff {err:.2e}")


# --- Test 1: norm_schur vs explicit projector, across partition shapes ---
# single row [4], single column [1,1,1,1], and mixed shapes in between.
sis = [0.7, 0.5, 0.3, 0.1]
sis = [s / math.sqrt(sum(x ** 2 for x in sis)) for s in sis]   # normalise sum(s_i^2)=1
k = 4
print(f"norm_schur vs explicit projector (k={k}, r={len(sis)}):")
for lam in [[4], [3, 1], [2, 2], [2, 1, 1], [1, 1, 1, 1]]:
    assert_close(norm_schur(lam, sis), norm_sq_projected_general(lam, sis), label=f"lam={lam}")

# --- Test 2: the f^lam s_lam values over all lam |- k sum to ||psi||^2 = 1 ---
k = 3
sis3 = [1 / math.sqrt(2), 1 / math.sqrt(3), 1 / math.sqrt(6)]
total = sum(norm_schur(list(lam), sis3) for lam in sage.Partitions(k).list())
assert_close(total, 1.0, label=f"sum over all partitions of {k} equals 1")


norm_schur vs explicit projector (k=4, r=4):
[PASS] lam=[4]
[PASS] lam=[3, 1]
[PASS] lam=[2, 2]
[PASS] lam=[2, 1, 1]
[PASS] lam=[1, 1, 1, 1]
[PASS] sum over all partitions of 3 equals 1


## 6. From pure states to mixed states: Schmidt number and the convex roof

Reference: Tóth, Moroder, Gühne, *Phys. Rev. Lett.* **114**, 160501 (2015).

Writing $\Pi^\lambda$ for the isotypic projector, the pure-state result gives
$$\|(\Pi^\lambda_{A_1\dots A_k}\otimes I_{B_1\dots B_k})\,|\psi_{AB}\rangle^{\otimes k}\|^2=0\iff\mathrm{SR}(|\psi_{AB}\rangle)<h(\lambda).$$
For mixed $\rho=\sum_i p_i|\psi_i\rangle\langle\psi_i|$, set
$\omega_{1\dots k}=\sum_i p_i(|\psi_i\rangle\langle\psi_i|)^{\otimes k}$. Then
$$\sum_i p_i\,\|(\Pi^\lambda\otimes I)|\psi_i\rangle^{\otimes k}\|^2=\operatorname{tr}\!\big[(\Pi^\lambda\otimes I)\,\omega_{1\dots k}\big],$$
and minimising over decompositions gives the exact convex-roof statement
$$\min_{\{p_i,\psi_i\}}\operatorname{tr}\!\big[(\Pi^\lambda\otimes I)\,\omega_{1\dots k}\big]=0\iff\mathrm{SN}(\rho)<h(\lambda).$$

### 6.1 SDP relaxation (symmetric extension)

The set of genuine $\omega=\sum_i p_i(|\psi_i\rangle\langle\psi_i|)^{\otimes k}$ is intractable, so we
relax it: $\omega\succeq0$, supported on the symmetric subspace, with the correct one-copy marginal
$\operatorname{tr}_{\neq1}\omega=\rho$, and PPT across the copy cuts. The relaxed set is a superset,
so the SDP optimum **lower-bounds** the convex roof:
$$\min_{\omega}\ \operatorname{tr}\!\big[(\Pi^\lambda_A\otimes I_B)\,\omega\big]\quad\text{s.t.}\quad\omega\succeq0,\ P_{\mathrm{sym}}\,\omega\,P_{\mathrm{sym}}=\omega,\ \operatorname{tr}_{\neq1}\omega=\rho,\ \omega^{T_S}\succeq0.$$
A strictly positive optimum certifies $\mathrm{SN}(\rho)\ge h(\lambda)$; a zero is inconclusive.

## Schmidt-number SDP (symmetric-extension relaxation)

The pure-state result above, $ | (\Pi^\lambda_A\otimes I_B) | \psi_{AB} \rangle^{ \otimes k} | ^2=f^\lambda s_\lambda(s_1^2,\dots,s_r^2)$,
vanishes **iff** $\mathrm{SR}(\psi)<h(\lambda)$. Extending to mixed $\rho$ via the convex roof gives, exactly,

$$\mathrm{SN}(\rho)<h(\lambda)\;\iff\;\min_{\omega}\ \mathrm{tr}\!\big[(\Pi^\lambda_A\otimes I_B)\,\omega\big]=0,$$

the minimum over $\omega=\sum_i p_i(|\psi_i\rangle\langle\psi_i|)^{\otimes k}$ with $\mathrm{tr}_{2\dots k}\omega=\rho$.
That set (convex hull of identical pure powers) is intractable, so we **relax** it to the
Tóth–Moroder–Gühne symmetric extension: $\omega\succeq0$, supported on $\mathrm{Sym}^k$,
with the right one-copy marginal, and PPT across the copy cuts. The relaxed set is a *superset*,
so the SDP optimum **lower-bounds** the convex roof:

$$\boxed{\;v^\star_\lambda>0\ \Longrightarrow\ \mathrm{SN}(\rho)\ge h(\lambda)\;}\qquad(v^\star_\lambda=0:\ \text{inconclusive}).$$

We work in the ordering $A_1\cdots A_k\,B_1\cdots B_k$, so the objective is just
$W=\Pi^\lambda_A\otimes I_{B^{\otimes k}}$ (a Kronecker product), while marginal/PPT act on the
$2k$ subsystems $[\,d_A,\dots,d_A,d_B,\dots,d_B\,]$. A copy is one $AB$ pair; a copy-permutation
acts *simultaneously* on the matching $A_j$ and $B_j$, so $V_\pi=P^A_\pi\otimes P^B_\pi$ and
$P_{\mathrm{sym}}=\frac1{k!}\sum_\pi V_\pi$.

### 6.2 Installing the SDP stack

We use PICOS with the cvxopt backend. Run once; if the import in the next cell fails, restart the
kernel and re-run.

In [16]:
import numpy as np
import picos as pic

def bose_projector(dA: int, dB: int, k: int) -> np.ndarray:
    """P_sym onto Sym^k of the k AB-copies, in the ordering A_1..A_k B_1..B_k.

    A copy-permutation acts jointly on the A-factors and the B-factors:
        V_pi = P^A_pi (x) P^B_pi,
    and P_sym = (1/k!) sum_pi V_pi. Reuses permutation_matrix_on_tensor_power.
    """
    Sk = sage.SymmetricGroup(k)
    D = (dA ** k) * (dB ** k)
    Psym = np.zeros((D, D), dtype=complex)
    for sigma in Sk:
        PA = permutation_matrix_on_tensor_power(sigma, dA, k)
        PB = permutation_matrix_on_tensor_power(sigma, dB, k)
        Psym += np.kron(PA, PB)
    Psym /= float(sage.factorial(k))
    return Psym


def schmidt_number_sdp(rho, lam: list[int], dA: int, dB: int,
                       ppt: bool = True, solver=None, tol: float = 1e-7):
    """Symmetric-extension SDP lower bound on the convex-roof value
           min_decomp  sum_i p_i || (Pi^lam_A (x) I_B) |psi_i>^{(x)k} ||^2 .

    Parameters
    ----------
    rho : density matrix on C^{dA} (x) C^{dB}, ordering |a>|b> -> a*dB + b.
    lam : partition of k = sum(lam); the test certifies SN(rho) >= h(lam) = len(lam).
    ppt : whether to add PPT constraints across the copy cuts.

    Returns
    -------
    (value, certified) with certified == (value > tol).
    value > tol  =>  SN(rho) >= h(lam).   value ~ 0  =>  inconclusive.
    """
    lam = sorted(lam, reverse=True)
    k = sum(lam)
    D = dA * dB

    PiA = isotypic_projector(lam, dA, k)             # Pi^lam on (C^dA)^{(x)k}
    W   = np.kron(PiA, np.eye(dB ** k))              # objective, ordering A_1..A_k B_1..B_k
    Psym = bose_projector(dA, dB, k)

    dims = [dA] * k + [dB] * k                       # 2k subsystems
    keep = [0, k]                                    # A_1 and B_1 -- one full AB copy
    trace_out = [i for i in range(2 * k) if i not in keep]

    P = pic.Problem()
    w = pic.HermitianVariable("w", (D ** k, D ** k))
    P.add_constraint(w >> 0)                                                   # PSD
    P.add_constraint(pic.Constant(Psym) * w * pic.Constant(Psym) == w)        # supported on Sym^k
    marg = pic.partial_trace(w, subsystems=trace_out, dimensions=dims)
    P.add_constraint(marg == pic.Constant(np.asarray(rho, dtype=complex)))     # tr_{2..k} w = rho
    if ppt:
        for r in range(1, k // 2 + 1):               # representative copy-cuts |S| = 1..floor(k/2)
            S = list(range(r))
            sub = S + [k + j for j in S]             # transpose A_j and B_j for j in S
            P.add_constraint(pic.partial_transpose(w, subsystems=sub, dimensions=dims) >> 0)

    P.set_objective("min", (pic.Constant(W) | w).real)   # <W, w> = tr(W w)
    P.solve(solver=solver, primals=True)
    val = float(P.value)
    return val, bool(val > tol)


### 6.3 Benchmark: isotropic states

For $\lambda=(1,\dots,1)$ of length $t$ the test certifies $\mathrm{SN}\ge t$. For $d\times d$
isotropic states the known threshold is $\mathrm{SN}\ge t\iff F>(t-1)/d$; with $d=2,\,t=2$ this is
$F>1/2$ (equivalently $p>1/3$).

In [17]:
# Benchmark: d x d isotropic states.  For lam = (1,...,1) of length t,
# the test certifies SN >= t.  Known threshold: SN >= t  <=>  fidelity F > (t-1)/d.
# Here d = 2, t = 2 :  SN >= 2 (entangled)  <=>  F > 1/2  (p > 1/3).

def isotropic(d: int, p: float):
    phi = np.zeros(d*d, dtype=complex)
    for i in range(d):
        phi[i*d + i] = 1/np.sqrt(d)
    Phi = np.outer(phi, phi.conj())
    rho = p*Phi + (1-p)*np.eye(d*d)/(d*d)
    return rho, phi

d = 2
print(f"{'p':>5} {'F':>6} {'SDP value':>12}   certify SN>=2 ?")
for p in [0.0, 0.20, 1/3, 0.34, 0.50, 0.80, 1.0]:
    rho, phi = isotropic(d, p)
    val, cert = schmidt_number_sdp(rho, [1, 1], dA=d, dB=d)
    F = float((phi.conj() @ rho @ phi).real)
    print(f"{p:5.2f} {F:6.3f} {val:12.3e}   {'YES' if cert else 'no'}")

print("\nExpected: certified exactly for F > 1/2  (p > 1/3).")

    p      F    SDP value   certify SN>=2 ?
 0.00  0.250   -3.283e-10   no
 0.20  0.400    1.728e-10   no
 0.33  0.500    2.884e-10   no
 0.34  0.505    2.500e-05   YES
 0.50  0.625    1.562e-02   YES
 0.80  0.850    1.225e-01   YES
 1.00  1.000    2.500e-01   YES

Expected: certified exactly for F > 1/2  (p > 1/3).


In [19]:
import picos as pic
print(pic.available_solvers())

['cvxopt']


In [18]:
import numpy as np

# ---------------------------------------------------------------------------
# Helpers to build test states
# ---------------------------------------------------------------------------

def isotropic(d: int, p: float):
    """Isotropic state: rho = p |Phi><Phi| + (1-p) I/d^2,  |Phi> = max. entangled."""
    phi = np.zeros(d * d, dtype=complex)
    for i in range(d):
        phi[i * d + i] = 1 / np.sqrt(d)
    Phi = np.outer(phi, phi.conj())
    rho = p * Phi + (1 - p) * np.eye(d * d) / (d * d)
    return rho, phi


def schmidt_rank_r_pure_state(d: int, r: int):
    """Pure state with exact Schmidt rank r, embedded in C^d (x) C^d  (r <= d)."""
    assert r <= d
    psi = np.zeros(d * d, dtype=complex)
    for i in range(r):
        psi[i * d + i] = 1 / np.sqrt(r)
    rho = np.outer(psi, psi.conj())
    return rho, psi


def product_state(d: int):
    """|0><0| (x) |0><0|."""
    psi = np.zeros(d * d, dtype=complex)
    psi[0] = 1.0
    return np.outer(psi, psi.conj()), psi


def separable_mixture(d: int):
    """(1/2)(|00><00| + |11><11|) -- classically correlated, SN = 1."""
    rho = np.zeros((d * d, d * d), dtype=complex)
    for i in range(min(2, d)):
        idx = i * d + i
        rho[idx, idx] += 0.5
    return rho


def maximally_mixed(d: int):
    return np.eye(d * d, dtype=complex) / (d * d)


def werner_state(d: int, p: float):
    """Werner state on C^d (x) C^d: rho = p * (2/(d(d-1))) * P_antisym + (1-p) * I/d^2
       Equivalent common parametrization: rho = (I - p*Flip)/(d^2 - p*d), kept simple here
       via the standard mixture with the swap operator.
       Known result: separable iff p <= 1/2 (for this convention with F = singlet-fidelity-like p).
       We use the textbook form: rho = (d - F) I/(d^3-d) + (F d -1)/(d^3-d) * d*Flip ... 
       -- to avoid convention confusion we instead build it directly from the flip operator.
    """
    Flip = np.zeros((d * d, d * d), dtype=complex)
    for i in range(d):
        for j in range(d):
            Flip[i * d + j, j * d + i] = 1.0
    rho = (np.eye(d * d) - p * Flip) / (d * d - p * d)
    return rho


# ---------------------------------------------------------------------------
# Test runner
# ---------------------------------------------------------------------------

def run_test(name, rho, dA, dB, expect_sn2, expect_sn3=None):
    val2, cert2 = schmidt_number_sdp(rho, [1, 1], dA=dA, dB=dB)
    line = f"{name:38s}  SN>=2: val={val2:9.3e} cert={str(cert2):5s} (expect {expect_sn2})"
    ok = (cert2 == expect_sn2)

    if expect_sn3 is not None:
        val3, cert3 = schmidt_number_sdp(rho, [1, 1, 1], dA=dA, dB=dB)
        line += f"  |  SN>=3: val={val3:9.3e} cert={str(cert3):5s} (expect {expect_sn3})"
        ok = ok and (cert3 == expect_sn3)

    print(line + ("   [OK]" if ok else "   [MISMATCH]"))
    return ok


print("=" * 110)
print("d = 2 tests")
print("=" * 110)

all_ok = True

rho, _ = maximally_mixed(2), None
all_ok &= run_test("maximally mixed (d=2)", maximally_mixed(2), 2, 2, expect_sn2=False)

all_ok &= run_test("product state |00><00|", product_state(2)[0], 2, 2, expect_sn2=False)

all_ok &= run_test("separable mixture (00+11)/2", separable_mixture(2), 2, 2, expect_sn2=False)

rho, _ = schmidt_rank_r_pure_state(2, 2)
all_ok &= run_test("pure Bell state (SR=2)", rho, 2, 2, expect_sn2=True)

# isotropic d=2 calibration: SN>=2 <=> p > 1/3  (F > 1/2)
for p in [0.0, 0.20, 1/3, 0.34, 0.50, 0.80, 1.0]:
    rho, _ = isotropic(2, p)
    expect = p > 1/3 + 1e-9
    all_ok &= run_test(f"isotropic d=2, p={p:.3f}", rho, 2, 2, expect_sn2=expect)

print()
print("=" * 110)
print("d = 3 tests  (SN can be 1, 2, or 3)")
print("=" * 110)

all_ok &= run_test("maximally mixed (d=3)", maximally_mixed(3), 3, 3,
                    expect_sn2=False, expect_sn3=False)

all_ok &= run_test("product state |00><00| (d=3)", product_state(3)[0], 3, 3,
                    expect_sn2=False, expect_sn3=False)

rho, _ = schmidt_rank_r_pure_state(3, 2)
all_ok &= run_test("pure state SR=2 (in d=3)", rho, 3, 3,
                    expect_sn2=True, expect_sn3=False)

rho, _ = schmidt_rank_r_pure_state(3, 3)
all_ok &= run_test("pure state SR=3 (max, d=3)", rho, 3, 3,
                    expect_sn2=True, expect_sn3=True)

# isotropic d=3 thresholds: SN>=t  <=>  F > (t-1)/d
# SN>=2 <=> F > 1/3   |   SN>=3 <=> F > 2/3
for p in [0.0, 0.30, 1/3 + 0.01, 0.50, 2/3 - 0.01, 2/3 + 0.01, 0.90, 1.0]:
    rho, phi = isotropic(3, p)
    F = float((phi.conj() @ rho @ phi).real)
    exp2 = F > 1/3 + 1e-9
    exp3 = F > 2/3 + 1e-9
    all_ok &= run_test(f"isotropic d=3, p={p:.3f} (F={F:.3f})", rho, 3, 3,
                        expect_sn2=exp2, expect_sn3=exp3)

print()
print("=" * 110)
print("Werner states (d=2) -- cross-check against independent entanglement criterion")
print("=" * 110)
# Werner state (this convention): separable iff p <= 1/2, entangled (and SN=2 since d=2) iff p > 1/2
for p in [0.0, 0.3, 0.5, 0.5 + 1e-3, 0.7, 1.0]:
    rho = werner_state(2, p)
    expect = p > 0.5 + 1e-9
    all_ok &= run_test(f"Werner d=2, p={p:.3f}", rho, 2, 2, expect_sn2=expect)

print()
print("ALL TESTS PASSED" if all_ok else "SOME TESTS FAILED -- check [MISMATCH] lines above")

d = 2 tests
maximally mixed (d=2)                   SN>=2: val=-3.283e-10 cert=False (expect False)   [OK]
product state |00><00|                  SN>=2: val=-3.670e-11 cert=False (expect False)   [OK]
separable mixture (00+11)/2             SN>=2: val=7.576e-14 cert=False (expect False)   [OK]
pure Bell state (SR=2)                  SN>=2: val=2.500e-01 cert=True  (expect True)   [OK]
isotropic d=2, p=0.000                  SN>=2: val=-3.283e-10 cert=False (expect False)   [OK]
isotropic d=2, p=0.200                  SN>=2: val=1.728e-10 cert=False (expect False)   [OK]
isotropic d=2, p=0.333                  SN>=2: val=2.884e-10 cert=False (expect False)   [OK]
isotropic d=2, p=0.340                  SN>=2: val=2.500e-05 cert=True  (expect True)   [OK]
isotropic d=2, p=0.500                  SN>=2: val=1.562e-02 cert=True  (expect True)   [OK]
isotropic d=2, p=0.800                  SN>=2: val=1.225e-01 cert=True  (expect True)   [OK]
isotropic d=2, p=1.000                  SN>=2: va

MemoryError: 

### Reading the output and scaling up

- **One-sided certificate.** A strictly positive optimum proves $\mathrm{SN}(\rho)\ge h(\lambda)$.
  A zero is *inconclusive* — it does not prove low Schmidt number, because the relaxed
  $\omega$ need not be a genuine mixture of identical pure powers (that gap is exactly the
  separability problem).
- **Which $(\lambda,k)$ to pick.** $h(\lambda)$ is the Schmidt number you test. At fixed $k$,
  the *shape* of $\lambda$ only changes the objective $W$ (same feasible set), so you can sweep
  all $\lambda\vdash k$ cheaply after building $P_{\mathrm{sym}}$ once. Taking $\lambda$ of the
  same height but **larger $k$** enlarges $\mathrm{Sym}^k$ and tightens the bound — a de Finetti
  hierarchy, monotonically stronger but more expensive. The single-column $\lambda=(1^t)$
  (antisymmetrizer) is the minimal, cleanest detector of $\mathrm{SN}\ge t$.
- **Cost.** The variable here is $D^k\times D^k$ with $D=d_Ad_B$, so it grows fast. To go beyond
  small cases: (i) restrict $\omega$ to the symmetric subspace via an isometry
  $V:\mathrm{Sym}^k(\mathbb C^D)\hookrightarrow(\mathbb C^D)^{\otimes k}$ and optimize over the
  $\binom{D+k-1}{k}\times\binom{D+k-1}{k}$ matrix $\sigma$ with $\omega=V\sigma V^\dagger$;
  and (ii) use a large-scale conic solver (MOSEK or SCS) instead of cvxopt.
- **Next experiment.** Sweep $d=3,k=2$ (threshold $F>1/3$ for $\mathrm{SN}\ge2$) and
  $d=3$, $\lambda=(1,1,1)$ (threshold $F>2/3$ for $\mathrm{SN}\ge3$); then check whether any
  $\lambda$ of other shapes at fixed $k$ certify states that the single-column (antisymmetric) choice misses —
  that is where this construction could beat existing moment / $k$-reduction witnesses.

## 7. Roadmap and open directions

**Problem.** Detect Schmidt rank across cuts of multipartite states using isotypic (Young)
projectors. The vanishing of $\Pi_\lambda$ on insufficiently repeated product states lets us read
entanglement dimensionality off the projected norm.

**Done.** The pure-state projected norm equals $f^\lambda s_\lambda(s_1^2,\dots,s_r^2)$ (Section 5),
recovering Christandl's $\|\cdot\|^2=0\iff\mathrm{SR}<h(\lambda)$ but carrying strictly more
information.

**In progress / planned.**
1. Relate the norm to a well-defined entanglement measure and its convex-roof extension (Schmidt
   number / cumulative Schmidt sums); cf. Tóth–Moroder–Gühne.
2. Lower-bound that measure with the symmetric-extension SDP of Section 6, adding the global PPT
   constraint. Open target: how large a Schmidt number can PPT states have under this test? Using
   many sectors $\lambda$, the construction may improve on existing witnesses — related to, but
   distinct from, Johnston's approach.
3. **Multipartite generalisation.** Schmidt rank across different bipartitions, using these
   techniques to detect incompatibility between cuts.
4. **Tensor rank** — a longer-term extension.

**Scaling.** Forming $\Pi_\lambda$ by summing over all $k!$ permutations is infeasible for large
$k$; an efficient symmetry-reduced test for whether the projection vanishes (and for the SDP) is
needed — e.g. restricting $\omega$ to the symmetric subspace via an isometry
$V:\mathrm{Sym}^k(\mathbb{C}^D)\hookrightarrow(\mathbb{C}^D)^{\otimes k}$ and using a large-scale
conic solver (MOSEK or SCS).